# Indoor67 — Data exploration
Original exploration by Spencer Kolodny (`859926a`), adapted with Codex to the shared workflow.
Use labeled training images only. Read the committed split; do not create another one.
Notebook outputs stay local; clear all outputs before committing.

In [ ]:
from pathlib import Path
import sys
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "src/dataset.py").is_file() and (p / "configs/baseline.yaml").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from the repository root or notebooks/.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Repository:", ROOT)

In [ ]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from PIL import Image
from IPython.display import display
from torch.utils.data import DataLoader
from src.dataset import SceneDataset, load_training_records
from src.models import baseline_weights, WEIGHT_NAME
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

## 1. Load and verify the shared manifests
Use `data/raw/train.csv` with `id,file_path,label`. All paths are relative to `data/raw/images/`.
The same loader verifies split/manifest hashes for CLI experiments and this notebook.

In [ ]:
DATA_DIR = ROOT / "data/raw"
IMAGE_ROOT = DATA_DIR / "images"
SPLIT_DIR = ROOT / "splits"
train_df, val_df, class_names, split_metadata = load_training_records(DATA_DIR, SPLIT_DIR)
df = pd.read_csv(DATA_DIR / "train.csv", dtype=str)
print(f"Images: {len(df):,}; classes: {len(class_names)}")
display(df.head())

## 2. Inspect class distribution

In [ ]:
counts = df.label.value_counts().sort_index()
print(counts.describe())
fig, ax = plt.subplots(figsize=(16, 5))
counts.plot.bar(ax=ax)
ax.set(title="Labeled training images per class", xlabel="Class", ylabel="Images")
plt.xticks(rotation=90); plt.tight_layout(); plt.show()

## 3. Visualize a reproducible sample

In [ ]:
sample = df.sample(n=min(12, len(df)), random_state=42)
fig, axes = plt.subplots(3, 4, figsize=(13, 9))
for ax, (_, row) in zip(axes.flat, sample.iterrows()):
    with Image.open(IMAGE_ROOT / row.file_path) as im:
        ax.imshow(im.convert("RGB"))
    ax.set_title(row.label, fontsize=9); ax.axis("off")
for ax in list(axes.flat)[len(sample):]: ax.axis("off")
plt.tight_layout(); plt.show()

## 4. Inspect image dimensions

In [ ]:
sizes = []
for p in df.file_path:
    with Image.open(IMAGE_ROOT / p) as im: sizes.append(im.size)
size_df = pd.DataFrame(sizes, columns=["width", "height"])
display(size_df.describe())
size_df.plot.scatter(x="width", y="height", alpha=0.25, figsize=(6, 4)); plt.show()

## 5. Reuse the team's saved stratified split
The 80/20 split and seed 42 were established before modeling. This cell inspects those
assignments; it never overwrites split files or writes `data/validation_split.csv`.

In [ ]:
assignments = pd.concat([train_df[["id"]].assign(split="train"), val_df[["id"]].assign(split="val")])
split_df = df.merge(assignments, on="id", validate="one_to_one")
assert len(split_df) == len(df)
assert set(train_df.id).isdisjoint(val_df.id)
print(split_df.split.value_counts())
display(pd.crosstab(split_df.label, split_df.split))
print("Split hashes:", split_metadata["split_sha256"])

## 6. Test a PyTorch batch

In [ ]:
# Class order is explicit: classes.csv, not inferred independently by each model.
transform = baseline_weights(WEIGHT_NAME).transforms()
train_dataset = SceneDataset(train_df, IMAGE_ROOT, class_names, transform)
val_dataset = SceneDataset(val_df, IMAGE_ROOT, class_names, transform)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=0,
                          generator=torch.Generator().manual_seed(SEED))
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=0)
for name, loader in [("Train", train_loader), ("Validation", val_loader)]:
    images, labels, ids = next(iter(loader))
    print(f"{name}: images {images.shape}, labels {labels.shape}, IDs {len(ids)}")
    assert images.shape[1:] == (3, 224, 224)
    assert torch.isfinite(images).all()
    assert labels.min() >= 0 and labels.max() < len(class_names)
print("Classes:", len(class_names))

## 7. Record what matters
Class counts, image dimensions, and the shared split are useful context for experiments.
Use `src.dataset.load_training_records` and `SceneDataset` in later work. Explain any
consequential choice in `docs/DECISIONS.md`; register runs with new IDs and preserve
historical results. Do not train on test images or manually recover their labels.